# 01 — Data Ingestion
**AutoAnalyst | Finance Module**

Handles: loading, inspection, type detection, ingestion report.
Output feeds into `02_data_cleaning.ipynb`

In [1]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
print('✅ Libraries loaded.')

✅ Libraries loaded.


In [2]:
# ── Cell 2: Config ───────────────────────────────────────────────────────────
# ONLY change DATASET_FILENAME when switching datasets
#
# Available:
#   RELIANCE.NS.csv
#   Credit card transactions - India - Simple.csv
#   Annual_P_L_1_final.csv  /  Annual_P_L_2_final.csv
#   Quarter_P_L_1_final.csv /  Quarter_P_L_2_final.csv
#   Balance_Sheet_final.csv
#   cash_flow_statments_final.csv
#   ratios_1_final.csv  /  ratios_2_final.csv
#   other_metrics_final.csv
#   price_final.csv  /  t1_prices.csv

DATASET_FILENAME = 'RELIANCE.NS.csv'
BASE_PATH        = r'R:\AutoAnalyst\finance_module\datasets'
DATASET_PATH     = os.path.join(BASE_PATH, DATASET_FILENAME)

print(f'Dataset  : {DATASET_FILENAME}')
print(f'Path     : {DATASET_PATH}')
print(f'Exists   : {os.path.exists(DATASET_PATH)}')

Dataset  : RELIANCE.NS.csv
Path     : R:\AutoAnalyst\finance_module\datasets\RELIANCE.NS.csv
Exists   : True


In [3]:
# ── Cell 3: Smart Loader ─────────────────────────────────────────────────────
# RELIANCE.NS.csv has 2 junk rows before real data.
# All other CSVs load normally.
# This function handles both cases automatically.

def smart_load(filepath):
    raw = pd.read_csv(filepath, nrows=3, header=0)
    first_val = str(raw.iloc[0, 0]).strip().lower()
    metadata_keywords = ['ticker', 'date', 'symbol', 'name', 'description']

    if first_val in metadata_keywords:
        df = pd.read_csv(filepath, skiprows=[1, 2], header=0)
        df.rename(columns={df.columns[0]: 'Date'}, inplace=True)
        print(f'[smart_load] Metadata rows skipped.')
    else:
        df = pd.read_csv(filepath, header=0)
        print(f'[smart_load] Clean header. Loaded normally.')

    return df


df = smart_load(DATASET_PATH)
print(f'Shape    : {df.shape}')
print(f'Columns  : {df.columns.tolist()}')

[smart_load] Metadata rows skipped.
Shape    : (1239, 6)
Columns  : ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']


In [4]:
# ── Cell 4: Basic Inspection ─────────────────────────────────────────────────

def basic_inspection(df, name):
    print('=' * 60)
    print(f'  INSPECTION REPORT — {name}')
    print('=' * 60)
    print(f'  Rows × Cols  : {df.shape[0]} × {df.shape[1]}')
    print(f'  Memory       : {df.memory_usage(deep=True).sum() / 1024:.2f} KB')
    print(f'  Duplicates   : {df.duplicated().sum()}')

    print(f'\n  Columns & dtypes:')
    for col in df.columns:
        print(f'    • {col}  ({df[col].dtype})')

    print(f'\n  First 5 rows:')
    display(df.head())

    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(f'\n  Missing values:')
    if missing.empty:
        print('    ✅ None')
    else:
        for col, cnt in missing.items():
            pct = cnt / len(df) * 100
            print(f'    • {col}: {cnt} ({pct:.2f}%)')

    print('=' * 60)


basic_inspection(df, DATASET_FILENAME)

  INSPECTION REPORT — RELIANCE.NS.csv
  Rows × Cols  : 1239 × 6
  Memory       : 70.31 KB
  Duplicates   : 0

  Columns & dtypes:
    • Date  (str)
    • Close  (float64)
    • High  (float64)
    • Low  (float64)
    • Open  (float64)
    • Volume  (int64)

  First 5 rows:


,Date,Close,High,Low,Open,Volume
0,2021-08-05,966.640076,975.947483,953.052502,957.604316,21252078
1,2021-08-06,946.168213,972.392146,941.503128,964.692519,16620987
2,2021-08-09,940.778442,946.507893,935.954862,942.522196,7494143
3,2021-08-10,945.715149,957.015494,939.238444,942.975016,11919198
4,2021-08-11,958.963074,960.185936,943.609165,949.746176,9184963



  Missing values:
    • Close: 1 (0.08%)
    • High: 1 (0.08%)
    • Low: 1 (0.08%)
    • Open: 1 (0.08%)


In [5]:
# ── Cell 5: Finance Type Detector ────────────────────────────────────────────
# Scores column names against keyword banks.
# Returns: timeseries | transactional | fundamental | unknown

def detect_finance_type(df):
    cols_str = ' '.join([c.lower().strip() for c in df.columns])

    timeseries_kw    = ['close', 'open', 'high', 'low', 'volume', 'price',
                        'adj close', 'ohlc']
    transactional_kw = ['transaction', 'amount', 'debit', 'credit', 'merchant',
                        'category', 'expense', 'card', 'payment', 'exp type']
    fundamental_kw   = ['revenue', 'profit', 'ebitda', 'eps', 'assets',
                        'liabilities', 'equity', 'net profit', 'bse', 'nse',
                        'market cap', 'ratio', 'roce', 'roe', 'sales',
                        'turnover', 'earnings']

    ts_score   = sum(1 for kw in timeseries_kw    if kw in cols_str)
    tx_score   = sum(1 for kw in transactional_kw if kw in cols_str)
    fn_score   = sum(1 for kw in fundamental_kw   if kw in cols_str)

    scores   = {'timeseries': ts_score, 'transactional': tx_score, 'fundamental': fn_score}
    best     = max(scores, key=scores.get)
    top2     = sorted(scores.values(), reverse=True)
    gap      = top2[0] - top2[1]

    if scores[best] == 0:
        return 'unknown', 'low', str(scores)
    elif scores[best] >= 3 and gap >= 2:
        conf = 'high'
    elif scores[best] >= 2 or gap >= 1:
        conf = 'medium'
    else:
        conf = 'low'

    return best, conf, str(scores)


dataset_type, confidence, reasoning = detect_finance_type(df)

print('=' * 60)
print('  TYPE DETECTION')
print('=' * 60)
print(f'  Type       : {dataset_type.upper()}')
print(f'  Confidence : {confidence.upper()}')
print(f'  Scores     : {reasoning}')
print('=' * 60)

  TYPE DETECTION
  Type       : TIMESERIES
  Confidence : HIGH
  Scores     : {'timeseries': 5, 'transactional': 0, 'fundamental': 0}


In [6]:
# ── Cell 6: Ingestion Report ─────────────────────────────────────────────────
# Packages everything into a dictionary.
# Pass df + ingestion_report into 02_data_cleaning.ipynb

ingestion_report = {
    'filename'     : DATASET_FILENAME,
    'filepath'     : DATASET_PATH,
    'shape'        : df.shape,
    'columns'      : list(df.columns),
    'dtypes'       : df.dtypes.astype(str).to_dict(),
    'missing_total': int(df.isnull().sum().sum()),
    'duplicates'   : int(df.duplicated().sum()),
    'dataset_type' : dataset_type,
    'confidence'   : confidence,
}

print('=' * 60)
print('  INGESTION REPORT')
print('=' * 60)
for k, v in ingestion_report.items():
    print(f'  {k:<18}: {v}')
print('=' * 60)
print('\n✅ Done. Ready for 02_data_cleaning.ipynb')

  INGESTION REPORT
  filename          : RELIANCE.NS.csv
  filepath          : R:\AutoAnalyst\finance_module\datasets\RELIANCE.NS.csv
  shape             : (1239, 6)
  columns           : ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']
  dtypes            : {'Date': 'str', 'Close': 'float64', 'High': 'float64', 'Low': 'float64', 'Open': 'float64', 'Volume': 'int64'}
  missing_total     : 4
  duplicates        : 0
  dataset_type      : timeseries
  confidence        : high

✅ Done. Ready for 02_data_cleaning.ipynb


In [7]:
# ── Cell 7: Sanity Check — All Datasets ──────────────────────────────────────
# Runs type detection across every dataset at once.
# Good verification before moving to cleaning.

ALL_DATASETS = [
    'RELIANCE.NS.csv',
    'Credit card transactions - India - Simple.csv',
    'Annual_P_L_1_final.csv',
    'Annual_P_L_2_final.csv',
    'Quarter_P_L_1_final.csv',
    'Quarter_P_L_2_final.csv',
    'Balance_Sheet_final.csv',
    'cash_flow_statments_final.csv',
    'ratios_1_final.csv',
    'ratios_2_final.csv',
    'other_metrics_final.csv',
    'price_final.csv',
    't1_prices.csv'
]

print(f'{"Dataset":<50} {"Type":<15} {"Confidence"}')
print('-' * 80)

for fname in ALL_DATASETS:
    fpath = os.path.join(BASE_PATH, fname)
    if not os.path.exists(fpath):
        print(f'{fname:<50} FILE NOT FOUND')
        continue
    try:
        tmp = smart_load(fpath)
        dtype, conf, _ = detect_finance_type(tmp)
        print(f'{fname:<50} {dtype:<15} {conf}')
    except Exception as e:
        print(f'{fname:<50} ERROR: {e}')

Dataset                                            Type            Confidence
--------------------------------------------------------------------------------
[smart_load] Metadata rows skipped.
RELIANCE.NS.csv                                    timeseries      high
[smart_load] Clean header. Loaded normally.
Credit card transactions - India - Simple.csv      transactional   high
[smart_load] Clean header. Loaded normally.
Annual_P_L_1_final.csv                             fundamental     high
[smart_load] Clean header. Loaded normally.
Annual_P_L_2_final.csv                             fundamental     high
[smart_load] Clean header. Loaded normally.
Quarter_P_L_1_final.csv                            fundamental     high
[smart_load] Clean header. Loaded normally.
Quarter_P_L_2_final.csv                            fundamental     high
[smart_load] Clean header. Loaded normally.
Balance_Sheet_final.csv                            fundamental     high
[smart_load] Clean header. Loaded nor

In [8]:
# Temp inspection cell — delete after use

# price_final
tmp1 = smart_load(os.path.join(BASE_PATH, 'price_final.csv'))
print("=== price_final ===")
print(tmp1.columns.tolist())
print(tmp1.head(3))

# Credit card - unique categories
tmp2 = smart_load(os.path.join(BASE_PATH, 'Credit card transactions - India - Simple.csv'))
print("\n=== Credit Card ===")
print(tmp2.dtypes)
print("Exp Type unique:", tmp2['Exp Type'].unique())
print("Card Type unique:", tmp2['Card Type'].unique())

# Annual P&L - all column names
tmp3 = smart_load(os.path.join(BASE_PATH, 'Annual_P_L_1_final.csv'))
print("\n=== Annual_P_L_1 columns ===")
print(tmp3.columns.tolist())

[smart_load] Clean header. Loaded normally.
=== price_final ===
['Name', 'BSE Code', 'NSE Code', 'Industry', 'Current Price', 'Return over 3months', 'Return over 6months', 'Volume 1month average', 'Volume 1week average', 'Volume', 'High price', 'Low price', 'High price all time', 'Low price all time', 'Return over 1day', 'Return over 1week', 'Return over 1month', 'DMA 50', 'DMA 200', 'DMA 50 previous day', 'DMA 200 previous day', 'RSI', 'MACD', 'MACD Previous Day', 'MACD Signal', 'MACD Signal Previous Day', 'Return over 1year', 'Return over 3years', 'Return over 5years', 'Volume 1year average', 'Return over 7years', 'Return over 10years', 'Market Capitalization', 'join_key']
               Name  BSE Code    NSE Code                    Industry  \
0        20 Microns  533022.0   20MICRONS  Mining / Minerals / Metals   
1  21st Cent. Mgmt.  526921.0  21STCENMGM       Finance & Investments   
2           360 ONE  542772.0      360ONE       Finance & Investments   

   Current Price  Retur